# Semantic abstraction graphicalizer

Thin experiment notebook: configure the run, invoke the package, and visualize the resulting graph, sample a connected subgraph, and narrate it.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    ExtractionDensityConfig,
    Graphicalizer,
    GraphicalizerConfig,
    NetworkXGraphStore,
    NodeEmbeddingConfig,
    NodeContextConfig,
    SubgraphNarrator,
    SubgraphNarrativeConfig,
    SubgraphNarrativePrompt,
    sample_random_connected_subgraph,
    graph_to_dot,
    load_ontology,
    load_corpus_articles,
    render_graph,
    validate_ontology_labeled_graph,
)
from IPython.display import SVG, display

## Configure and run

In [ ]:
ASSETS_ROOT = PROJECT_ROOT / 'assets'
CORPUS_PATH = PROJECT_ROOT / 'outputs' / 'pubmed_screening' / 'corpus_articles.parquet'
CORPUS_PATHOGENS = None  # e.g. ['Nipah virus']
CORPUS_START_YEAR = None
CORPUS_END_YEAR = None
PUBMED_ABSTRACT_INDEX = 0  # zero-based index after corpus filtering
CORPUS_ARTICLES = load_corpus_articles(
    CORPUS_PATH,
    pathogens=CORPUS_PATHOGENS,
    start_year=CORPUS_START_YEAR,
    end_year=CORPUS_END_YEAR,
)
if CORPUS_ARTICLES.empty:
    raise ValueError('No abstracts remain after corpus filtering.')
if not 0 <= PUBMED_ABSTRACT_INDEX < len(CORPUS_ARTICLES):
    raise IndexError(
        f'PUBMED_ABSTRACT_INDEX must be between 0 and {len(CORPUS_ARTICLES) - 1}.'
    )
ABSTRACT_ROW = CORPUS_ARTICLES.iloc[PUBMED_ABSTRACT_INDEX]
ABSTRACT_REF = f'{ABSTRACT_ROW["pathogen"]}/{ABSTRACT_ROW["pmid"]}'
ASSEMBLED_ONTOLOGY_PATH = ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology_assembled.yaml'
ONTOLOGY_PATH = (
    ASSEMBLED_ONTOLOGY_PATH
    if ASSEMBLED_ONTOLOGY_PATH.exists()
    else ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology.yaml'
)
PROMPT_PATH = ASSETS_ROOT / 'prompts' / 'graphicalizer_prompt_template.yaml'
PROMPT_SNAPSHOT_PATH = PROJECT_ROOT / 'outputs' / 'prompts' / 'ontology-aware-graphicalizer-0.4.0.yaml'
GRAPH_OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'selected_ontology_graph.svg'
LLM_PROVIDER = 'openai'  # choose 'openai' or 'ollama'
LLM_MODEL = 'gpt-4o-mini' if LLM_PROVIDER == 'openai' else 'gemma4:12b-mlx' # 'gpt-4o-mini' 'gpt-5-nano'
LLM_OPTIONS = (
    {'max_output_tokens': 16384}
    if LLM_PROVIDER == 'openai'
    else {'num_ctx': 32768, 'num_predict': -1}
)
NARRATIVE_COLUMNS = 80
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
from sentence_transformers import SentenceTransformer
EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME)
EMBEDDING_CONFIG = NodeEmbeddingConfig(
    normalize=True,
    model_id=EMBEDDING_MODEL_NAME,
)

abstract_text = str(ABSTRACT_ROW['abstract'])
print('Selected corpus row:', ABSTRACT_REF)
print(textwrap.fill(abstract_text, width=NARRATIVE_COLUMNS))

ontology = load_ontology(ONTOLOGY_PATH)
ENTITY_EFFORT = 0.05
RELATION_EFFORT = 1.5
MINIMUM_ENTITY_FRACTION = 0.75
DENSITY_RETRIES = 2
density = ExtractionDensityConfig(
    entities_per_word=ENTITY_EFFORT,
    relations_per_entity=RELATION_EFFORT,
    minimum_entity_fraction=MINIMUM_ENTITY_FRACTION,
    density_retries=DENSITY_RETRIES,
)
context = NodeContextConfig(
    max_sentences=3,
    max_evidence_items=2,
    include_evidence=True,
    include_uncertainty=True,
)
config = GraphicalizerConfig(
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    extraction_density=density,
    node_context=context,
    prompt_template_path=PROMPT_PATH,
    prompt_snapshot_path=PROMPT_SNAPSHOT_PATH,
    disconnected_policy='largest_component',
    context_policy='all_nodes',
    casting_retries=2,
)

graphicalizer = Graphicalizer.from_provider(
    ontology,
    config,
    options=LLM_OPTIONS,
    embedding_model=EMBEDDING_MODEL,
    embedding_config=EMBEDDING_CONFIG,
)
result = graphicalizer.run(abstract_text)

In [ ]:
print('Model:', config.model)
print('Target entities:', density.target_counts(abstract_text)['target_entity_count'])
print('Minimum entities:', density.target_counts(abstract_text)['minimum_entity_count'])
print('Source SHA-256:', result.run_metadata['source_sha256'])
print('Raw validation:', result.raw_validation.to_dict())
print('Normalized validation:', result.normalized_validation.to_dict())
print('Normalization report:', result.normalization_report)
print('Final validation:', result.final_validation.to_dict())
print('Entities:', len(result.normalized_extraction.entities))
print('Relations:', len(result.ontology_casting.relations))
print('Node contexts:', len(result.node_contexts))
print('Embedding metadata:', result.graph.graph.get('node_context_embeddings'))

In [ ]:
GRAPH_STORE = NetworkXGraphStore(PROJECT_ROOT / 'outputs' / 'graphs')
GRAPH_ID = f'pubmed_{ABSTRACT_ROW["pathogen"]}_{ABSTRACT_ROW["pmid"]}'
GRAPH_PATH = GRAPH_STORE.save(result.graph, GRAPH_ID)
LOADED_GRAPH = GRAPH_STORE.load(GRAPH_ID)
print('Saved graph:', GRAPH_PATH)
print('Stored graphs:', len(GRAPH_STORE.list()))
print('Loaded nodes:', LOADED_GRAPH.number_of_nodes())
print('Loaded edges:', LOADED_GRAPH.number_of_edges())

## Visualize the typed graph

In [ ]:
validate_ontology_labeled_graph(result.typed_graph)
dot_text = graph_to_dot(
    result.typed_graph,
    graph_name='SelectedPubMedGraph',
    require_ontology_labels=True,
)
print(dot_text)
render_graph(
    result.typed_graph,
    GRAPH_OUTPUT_PATH,
    graph_name='SelectedPubMedGraph',
    require_ontology_labels=True,
)
display(SVG(filename=str(GRAPH_OUTPUT_PATH)))

## Sample and narrate a connected subgraph

In [ ]:
SUBGRAPH_NODE_COUNT = 3
SUBGRAPH_SEED = None
NARRATIVE_WORDS = 100
SUBGRAPH_OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'selected_random_subgraph.svg'
NARRATIVE_PROMPT_PATH = ASSETS_ROOT / 'prompts' / 'subgraph_narrative_prompt_template.yaml'
NARRATIVE_SNAPSHOT_PATH = PROJECT_ROOT / 'outputs' / 'prompts' / 'subgraph-narrator-0.1.0.yaml'

sampled_subgraph = sample_random_connected_subgraph(
    result.typed_graph,
    num_nodes=SUBGRAPH_NODE_COUNT,
    seed=SUBGRAPH_SEED,
)
validate_ontology_labeled_graph(sampled_subgraph)
render_graph(
    sampled_subgraph,
    SUBGRAPH_OUTPUT_PATH,
    graph_name='SelectedRandomSubgraph',
    require_ontology_labels=True,
)
display(SVG(filename=str(SUBGRAPH_OUTPUT_PATH)))

In [ ]:

narrator_prompt = SubgraphNarrativePrompt.from_yaml_file(NARRATIVE_PROMPT_PATH)
narrator = SubgraphNarrator.from_graphicalizer(
    graphicalizer,
    prompt=narrator_prompt,
    prompt_snapshot_path=NARRATIVE_SNAPSHOT_PATH,
)
narrative = narrator.narrate(
    sampled_subgraph,
    SubgraphNarrativeConfig(target_words=NARRATIVE_WORDS),
)
print(textwrap.fill(narrative.narrative, width=NARRATIVE_COLUMNS))
print('Requested words:', narrative.requested_words)
print('Actual words:', narrative.actual_words)
if narrative.uncertainty:
    print('Uncertainty:', narrative.uncertainty)


## Inspect node contexts

In [ ]:
for node_context in result.node_contexts:
    print(f'\n[{node_context.entity_id}] {textwrap.fill(node_context.summary, width=NARRATIVE_COLUMNS)}')
    for evidence in node_context.evidence:
        print(f'  evidence: {textwrap.fill(evidence, width=NARRATIVE_COLUMNS)}')
    if node_context.uncertainty:
        print(f'  uncertainty: {textwrap.fill(node_context.uncertainty, width=NARRATIVE_COLUMNS)}')